# Agent 1: CRM Correlator

**Purpose**: Score raw sensor readings from multiple modalities (vision, e-nose, e-tongue) against Certified Reference Materials (CRM) and output enriched samples with:
- Per-sample match scores against all CRMs
- Top candidate and confidence
- Differentiators applied (if ambiguous)
- Cross-modal reasoning

**Output**: `enriched_samples.json` → Input for Agent 2 (Batch Risk Assessor)

## Step 1: Setup & Imports

In [ ]:
import os
import sys
import json
import time
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, asdict
from enum import Enum
import logging

# LangGraph imports
from langgraph.graph import StateGraph, START, END
from langgraph.types import StateSnapshot
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

# Add foodguard lib to path
sys.path.insert(0, '..')
from foodguard_lib import (
    generate_id, insert_agent_execution, generate_batch_id, CRM_PATH, PROMPT_PATH,
    ENRICHED_SAMPLES_PATH
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Imports successful")

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Configure vLLM environment
os.environ["BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPENAI_API_KEY"] = "abc-123"

# Initialize LLM client for enhanced reasoning
try:
    llm = ChatOpenAI(
        model="Qwen3-4B",
        base_url=os.environ["BASE_URL"],
        api_key=os.environ["OPENAI_API_KEY"],
        temperature=0.7,
        max_tokens=1024,
    )
    print("✓ vLLM (Qwen3-4B) configured and ready")
except Exception as e:
    print(f"⚠ Warning: Could not connect to vLLM server: {e}")
    print("  Make sure vLLM is running: vllm serve Qwen/Qwen3-4B --port 8000 --api-key abc-123")
    llm = None

## Step 2: Load CRM Reference & System Prompt

In [ ]:
with open(CRM_PATH, 'r') as f:
    crm_data = json.load(f)

crms = crm_data['crms']
print(f"✓ Loaded {len(crms)} CRM references")
print(f"  CRM IDs: {[c['crm_id'] for c in crms]}")

# Load system prompt
with open(PROMPT_PATH, 'r') as f:
    prompt_content = f.read()

# Extract SYSTEM_PROMPT from the file
import re
system_prompt_match = re.search(r'SYSTEM_PROMPT = """(.+?)"""', prompt_content, re.DOTALL)
if system_prompt_match:
    SYSTEM_PROMPT = system_prompt_match.group(1)
    print(f"✓ Loaded system prompt ({len(SYSTEM_PROMPT)} chars)")
else:
    print("✗ Could not extract SYSTEM_PROMPT from prompt file")

In [ ]:
# ========== LLM-Enhanced CRM Reasoning ==========

def use_llm_for_crm_reasoning(sample_data: Dict, top_matches: List[Dict], modalities: List) -> Dict:
    """
    Use vLLM (Qwen3-4B) to provide intelligent reasoning for CRM matching.
    Enhances confidence scores and provides cross-modal validation.
    """
    if llm is None:
        return {}

    prompt = f"""
You are a food safety expert analyzing milk samples using multimodal sensor data.

SAMPLE DATA:
{json.dumps(sample_data, indent=2)}

TOP CRM MATCHES:
{json.dumps(top_matches[:3], indent=2)}

MODALITIES ANALYZED: {', '.join(modalities)}

TASK:
1. Analyze the sample data against the CRM matches
2. Consider cross-modal consistency (vision, e-nose, e-tongue)
3. Provide a confidence-adjusted recommendation
4. Flag any anomalies or ambiguities

RESPONSE FORMAT:
{{
    "recommended_crm": "<crm_id>",
    "confidence_adjustment": <-0.1 to +0.1>,
    "reasoning": "brief explanation",
    "cross_modal_consistency": "high|medium|low",
    "anomalies": ["list of concerns if any"]
}}

Analyze and respond:
"""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        result = json.loads(response.content)
        return result
    except Exception as e:
        print(f"LLM reasoning failed: {e}")
        return {}

print("✓ LLM reasoning functions defined")

## Step 3: Define Agent State & Data Classes

In [ ]:
@dataclass
class CRMCorrelatorInput:
    """Input state for CRM Correlator agent."""
    raw_samples: List[Dict[str, Any]]  # List of sensor readings
    modalities: List[str]               # Which modalities present: [vision, enose, etongue]
    batch_id: str                       # Batch identifier

@dataclass
class CRMCorrelatorOutput:
    """Output state from CRM Correlator agent."""
    enriched_samples: List[Dict[str, Any]]  # Samples with CRM match scores & top candidate
    batch_id: str
    agent_execution_id: str
    reasoning: str
    execution_time_ms: float

# LangGraph state type
class CRMCorrelatorState(dict):
    """State dict for LangGraph node."""
    pass

print("✓ State classes defined")

## Step 4: CRM Matching & Scoring Logic

In [ ]:
def score_sample_against_crms(
    sample: Dict[str, Any],
    crms: List[Dict[str, Any]],
    modalities: List[str]
) -> Dict[str, float]:
    """
    Score a sample against all CRMs.
    Returns dict: {crm_certified_class: match_score}
    """
    scores = {}

    for crm in crms:
        crm_class = crm['certified_class']
        certified_ranges = crm['certified_ranges']

        total_features_checked = 0
        features_matched = 0

        # Check each modality present in sample
        for modality in modalities:
            if modality not in certified_ranges:
                continue

            modality_ranges = certified_ranges[modality]

            for feature_name, [min_val, max_val] in modality_ranges.items():
                if feature_name in sample:
                    total_features_checked += 1
                    sample_val = sample[feature_name]

                    # Check if value is within CRM range
                    if min_val <= sample_val <= max_val:
                        features_matched += 1

        # Calculate match score
        if total_features_checked > 0:
            match_score = features_matched / total_features_checked
        else:
            match_score = 0.0

        scores[crm_class] = round(match_score, 2)

    return scores

print("✓ Scoring function defined")

## Step 5: Differentiator Logic (for ambiguous cases)

In [ ]:
def apply_differentiators(
    sample: Dict[str, Any],
    top_candidate: str,
    second_candidate: str,
    crm_scores: Dict[str, float]
) -> tuple[str, str]:  # (resolved_candidate, differentiator_used)
    """
    Apply cross-modal differentiators to resolve ambiguous cases.
    Returns: (final_candidate, differentiator_description)
    """

    # DIFF-1: urea_added vs ammonium_sulfate
    if {top_candidate, second_candidate} == {'urea_added', 'ammonium_sulfate'}:
        sulfur_signal = sample.get('sulfur_signal')
        crystal_presence = sample.get('crystal_presence')

        if (sulfur_signal and sulfur_signal > 0.35) or (crystal_presence and crystal_presence > 0.50):
            return 'ammonium_sulfate', 'DIFF-1: High sulfur/crystal → ammonium_sulfate'
        else:
            return 'urea_added', 'DIFF-1: Low sulfur/crystal → urea_added'

    # DIFF-2: oil_surfactant vs formalin_added
    if {top_candidate, second_candidate} == {'oil_surfactant', 'formalin_added'}:
        alcohol_signal = sample.get('alcohol_signal')
        sourness = sample.get('sourness')

        if (alcohol_signal and alcohol_signal > 0.80) or (sourness and sourness > 2.5):
            return 'oil_surfactant', 'DIFF-2: High alcohol/sourness → oil_surfactant'
        elif (alcohol_signal and alcohol_signal < 0.20) or (sourness and sourness < 0.6):
            return 'formalin_added', 'DIFF-2: Low alcohol/sourness → formalin_added'

    # DIFF-3: spoiled vs urea_added
    if {top_candidate, second_candidate} == {'spoiled', 'urea_added'}:
        alcohol_signal = sample.get('alcohol_signal')
        sourness = sample.get('sourness')

        if (alcohol_signal and alcohol_signal > 1.50) or (sourness and sourness > 5.0):
            return 'spoiled', 'DIFF-3: High alcohol/sourness → spoiled'
        else:
            return 'urea_added', 'DIFF-3: Low alcohol/sourness → urea_added'

    # DIFF-4: spoiled vs formalin_added
    if {top_candidate, second_candidate} == {'spoiled', 'formalin_added'}:
        bacterial_markers = sample.get('bacterial_markers')
        alcohol_signal = sample.get('alcohol_signal')
        plate_presence = sample.get('plate_presence')

        if (bacterial_markers and bacterial_markers > 0.60) or (alcohol_signal and alcohol_signal > 1.50):
            return 'spoiled', 'DIFF-4: High bacteria/alcohol → spoiled'
        elif (plate_presence and plate_presence > 0.70) or (alcohol_signal and alcohol_signal < 0.20):
            return 'formalin_added', 'DIFF-4: High plate/low alcohol → formalin_added'

    # DIFF-5: water_diluted vs authentic
    if {top_candidate, second_candidate} == {'water_diluted', 'authentic'}:
        globule_density = sample.get('globule_density')
        sweetness = sample.get('sweetness')

        if (globule_density and globule_density < 0.55) or (sweetness and sweetness < 4.0):
            return 'water_diluted', 'DIFF-5: Low density/sweetness → water_diluted'
        else:
            return 'authentic', 'DIFF-5: High density/sweetness → authentic'

    # No differentiator applies
    return top_candidate, 'none — resolved by score margin'

print("✓ Differentiator logic defined")

## Step 6: Confidence Calculation

In [ ]:
def calculate_confidence(
    top_score: float,
    second_score: float,
    num_modalities: int,
    differentiator_used: str
) -> float:
    """
    Calculate confidence score with modality-based adjustments.
    """
    confidence = top_score

    # Adjust by modality count
    if num_modalities == 1:
        confidence = min(confidence, 0.65)
    elif num_modalities == 2:
        confidence = min(confidence, 0.80)
    # 3+ modalities: no cap

    # Adjust if differentiator was used
    if 'none' not in differentiator_used.lower():
        confidence -= 0.05

    # Adjust if top-2 were close
    if abs(top_score - second_score) < 0.10:
        confidence -= 0.05

    # Cap at 0.95 and ensure >= 0.0
    confidence = min(max(confidence, 0.0), 0.95)

    return round(confidence, 2)

print("✓ Confidence calculation defined")

## Step 7: Risk & Status Mapping

In [ ]:
RISK_STATUS_MAP = {
    'authentic': {'risk_level': 'None', 'status': 'Safe'},
    'water_diluted': {'risk_level': 'Medium', 'status': 'Caution'},
    'urea_added': {'risk_level': 'High', 'status': 'Unsafe'},
    'ammonium_sulfate': {'risk_level': 'High', 'status': 'Unsafe'},
    'oil_surfactant': {'risk_level': 'High', 'status': 'Unsafe'},
    'formalin_added': {'risk_level': 'Critical', 'status': 'Unsafe'},
    'spoiled': {'risk_level': 'Critical', 'status': 'Unsafe'},
    'inconclusive': {'risk_level': 'Unknown', 'status': 'Inconclusive'},
}

def get_risk_status(adulterant_class: str) -> Dict[str, str]:
    return RISK_STATUS_MAP.get(adulterant_class, {'risk_level': 'Unknown', 'status': 'Inconclusive'})

print("✓ Risk/Status mapping defined")

## Step 8: Enrich Single Sample

In [ ]:
def enrich_sample(
    sample: Dict[str, Any],
    crm_scores: Dict[str, float],
    modalities: List[str]
) -> Dict[str, Any]:
    """
    Enrich a single sample with CRM matching results.
    """
    # Sort by score descending
    sorted_scores = sorted(crm_scores.items(), key=lambda x: x[1], reverse=True)
    top_candidate = sorted_scores[0][0]
    top_score = sorted_scores[0][1]
    second_candidate = sorted_scores[1][0] if len(sorted_scores) > 1 else None
    second_score = sorted_scores[1][1] if len(sorted_scores) > 1 else 0.0

    # Handle inconclusive cases
    if top_score < 0.40:
        final_candidate = 'inconclusive'
        differentiator_used = 'N/A — low all scores'
        confidence = 0.0
    # Apply differentiators if ambiguous
    elif top_score >= 0.80 and second_score < 0.60:
        # Clear winner
        final_candidate = top_candidate
        differentiator_used = 'none — clear winner'
        confidence = calculate_confidence(top_score, second_score, len(modalities), differentiator_used)
    elif second_candidate and abs(top_score - second_score) <= 0.15:
        # Ambiguous — apply differentiators
        final_candidate, differentiator_used = apply_differentiators(sample, top_candidate, second_candidate, crm_scores)
        confidence = calculate_confidence(top_score, second_score, len(modalities), differentiator_used)
    else:
        # Candidate resolved by margin
        final_candidate = top_candidate
        differentiator_used = 'none — resolved by score margin'
        confidence = calculate_confidence(top_score, second_score, len(modalities), differentiator_used)

    # Get risk/status
    risk_status = get_risk_status(final_candidate)

    # Find matched CRM
    matched_crm = None
    reference_methods = []
    for crm in crms:
        if crm['certified_class'] == final_candidate:
            matched_crm = crm
            reference_methods = crm.get('reference_methods', [])
            break

    matched_crm_id = matched_crm['crm_id'] if matched_crm else 'N/A'

    # Construct enriched sample
    enriched = {
        'sample_id': sample.get('sample_id', 'unknown'),
        'modalities_provided': modalities,
        'crm_scores': crm_scores,
        'top_candidate': top_candidate,
        'second_candidate': second_candidate,
        'differentiator_used': differentiator_used,
        'ground_truth': final_candidate,
        'confidence': confidence,
        'matched_crm_id': matched_crm_id,
        'reference_methods_applied': reference_methods,
        'risk_level': risk_status['risk_level'],
        'status': risk_status['status'],
        'note': f"Sample {sample.get('sample_id')} matched {matched_crm_id} with {confidence:.0%} confidence via {differentiator_used}."
    }

    return enriched

print("✓ Sample enrichment function defined")

## Step 9: LangGraph Node Function

In [ ]:
def crm_correlator_node(state: CRMCorrelatorState) -> CRMCorrelatorState:
    """
    Agent 1 node: CRM Correlator.
    Processes raw samples, scores against CRMs, applies differentiators,
    and outputs enriched samples.
    """
    start_time = time.time()

    raw_samples = state.get('raw_samples', [])
    modalities = state.get('modalities', [])
    batch_id = state.get('batch_id', 'unknown')
    investigation_id = state.get('investigation_id')

    logger.info(f"CRM Correlator processing {len(raw_samples)} samples from modalities: {modalities}")

    enriched_samples = []

    for sample in raw_samples:
        # Score sample against all CRMs
        crm_scores = score_sample_against_crms(sample, crms, modalities)

        # Enrich sample with matching results
        enriched = enrich_sample(sample, crm_scores, modalities)
        enriched_samples.append(enriched)

    # Log execution
    execution_time = (time.time() - start_time) * 1000  # ms
    exec_id = generate_id('EXEC')

    reasoning = f"Scored {len(raw_samples)} samples against {len(crms)} CRMs. " \
                f"Applied cross-modal differentiators for ambiguous cases. " \
                f"Execution time: {execution_time:.1f}ms"

    # Log to database if investigation_id available
    if investigation_id:
        try:
            insert_agent_execution(
                investigation_id=investigation_id,
                agent_name="CRM Correlator",
                input_data={'raw_samples_count': len(raw_samples), 'modalities': modalities},
                output_data={'enriched_samples_count': len(enriched_samples)},
                reasoning=reasoning,
                execution_time_ms=execution_time
            )
            logger.info(f"[DB] Agent execution logged: {exec_id}")
        except Exception as e:
            logger.warning(f"[DB] Failed to log execution: {e}")

    # Return updated state
    state['enriched_samples'] = enriched_samples
    state['agent_execution_id'] = exec_id
    state['reasoning'] = reasoning
    state['execution_time_ms'] = execution_time
    state['batch_id'] = batch_id

    logger.info(f"CRM Correlator complete: {len(enriched_samples)} samples enriched")

    return state

## Step 10: Build LangGraph

In [ ]:
# Build graph
builder = StateGraph(CRMCorrelatorState)
builder.add_node("correlator", crm_correlator_node)
builder.set_entry_point("correlator")
builder.add_edge("correlator", END)

graph = builder.compile()

print("✓ LangGraph compiled")
print("\nGraph structure:")
print(graph.get_graph().draw_ascii())

## Step 11: Test with Sample Data

In [ ]:
# Create test samples (one from each modality + combined)
test_samples = [
    {
        'sample_id': 'S001',
        'ammonia_signal': 0.15,
        'alcohol_signal': 0.5,
        'sulfur_signal': 0.1,
        'voc_signal': 1.0,
        'hydrocarbon_signal': 0.3,
        'aromatic_signal': 0.2,
    },
    {
        'sample_id': 'S002',
        'globule_density': 0.95,
        'globule_roundness': 0.95,
        'cluster_factor': 0.05,
        'crystal_presence': 0.02,
        'blob_irregularity': 0.05,
        'plate_presence': 0.02,
        'bacterial_markers': 0.02,
    },
    {
        'sample_id': 'S003',
        'sweetness': 5.0,
        'saltiness': 1.5,
        'sourness': 1.0,
        'bitterness': 0.3,
        'umami': 1.0,
        'astringency': 0.2,
    },
]

# Run graph
initial_state = {
    'raw_samples': test_samples,
    'modalities': ['enose', 'vision', 'etongue'],
    'batch_id': generate_batch_id(),
}

output_state = graph.invoke(initial_state)

print(f"\n✓ Graph executed successfully")
print(f"Batch ID: {output_state['batch_id']}")
print(f"Samples enriched: {len(output_state['enriched_samples'])}")
print(f"Execution time: {output_state['execution_time_ms']:.1f}ms")

## Step 12: Display Results

In [ ]:
# Pretty-print enriched samples
print("\n" + "="*80)
print("ENRICHED SAMPLES OUTPUT")
print("="*80)

for i, sample in enumerate(output_state['enriched_samples'], 1):
    print(f"\n[Sample {i}] {sample['sample_id']}")
    print(f"  Ground Truth:      {sample['ground_truth']}")
    print(f"  Top Candidate:     {sample['top_candidate']}")
    print(f"  Confidence:        {sample['confidence']:.0%}")
    print(f"  Risk Level:        {sample['risk_level']}")
    print(f"  Status:            {sample['status']}")
    print(f"  Differentiator:    {sample['differentiator_used']}")
    print(f"  Matched CRM:       {sample['matched_crm_id']}")
    print(f"  Note:              {sample['note']}")

## Step 13: Save Enriched Samples to JSON

In [ ]:
# Save enriched samples
output_path = ENRICHED_SAMPLES_PATH
os.makedirs(os.path.dirname(output_path), exist_ok=True)

output_json = {
    'batch_id': output_state['batch_id'],
    'agent': 'CRM_Correlator',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'samples_count': len(output_state['enriched_samples']),
    'modalities': output_state.get('modalities', []),
    'samples': output_state['enriched_samples'],
}

with open(output_path, 'w') as f:
    json.dump(output_json, f, indent=2)

print(f"✓ Enriched samples saved to: {output_path}")
print(f"  File size: {os.path.getsize(output_path) / 1024:.1f} KB")